In [11]:
# Carregamento do dataset
import os
import numpy as np
from scipy.io import loadmat

PASTA_PACIENTE = "./VP001-EEG"

# Carrega o sinal contínuo (cnt) e os marcadores de eventos (mrk)
mat_cnt = loadmat(os.path.join(PASTA_PACIENTE, 'cnt_nback.mat'), struct_as_record=False, squeeze_me=True)
mat_mrk = loadmat(os.path.join(PASTA_PACIENTE, 'mrk_nback.mat'), struct_as_record=False, squeeze_me=True)

# Acessa as estruturas internas
chave_cnt = [k for k in mat_cnt.keys() if not k.startswith('__')][0]
chave_mrk = [k for k in mat_mrk.keys() if not k.startswith('__')][0]

cnt = mat_cnt[chave_cnt]
mrk = mat_mrk[chave_mrk]

print("Chaves encontradas em cnt:", [k for k in mat_cnt.keys() if not k.startswith('__')])
print("Chaves encontradas em mrk:", [k for k in mat_mrk.keys() if not k.startswith('__')])

Chaves encontradas em cnt: ['cnt_nback']
Chaves encontradas em mrk: ['mrk_nback']


In [12]:
# Verifica informações básicas do dataset

# Taxa de amostragem
fs = float(cnt.fs)
print(f'Taxa de amostragem: {fs} Hz')

# Canais utilizados
channels = list(cnt.clab)
print('\nCanais utilizados: ', channels)
print('Número de canais: ', len(channels))

Taxa de amostragem: 200.0 Hz

Canais utilizados:  ['Fp1', 'AFF5h', 'AFz', 'F1', 'FC5', 'FC1', 'T7', 'C3', 'Cz', 'CP5', 'CP1', 'P7', 'P3', 'Pz', 'POz', 'O1', 'Fp2', 'AFF6h', 'F2', 'FC2', 'FC6', 'C4', 'T8', 'CP2', 'CP6', 'P4', 'P8', 'O2', 'HEOG', 'VEOG']
Número de canais:  30


In [13]:
# Prepara os dados principais

# X: Matriz contínua de sinais
X = cnt.x

# trials: momento da amostragem onde cada estímulo começa
if mrk.time.max() > X.shape[0]:
    trials = np.round(mrk.time * fs / 1000.0).astype(int)
else:
    trials = mrk.time.astype(int)

# y: rótulos das classes
if hasattr(mrk, 'y'):
    if mrk.y.ndim > 1:
        y = np.argmax(mrk.y, axis=0)
    else:
        y = mrk.y.astype(int)
else:
    y = np.zeros(len(trials), dtype=int)

# Tabela com [amostra_inicio, classe]
events = np.column_stack([trials, y])

print(X.shape)
print(y.shape)
print(trials.shape)
print(events.shape)

(394930, 30)
(567,)
(567,)
(567, 2)


In [14]:
print("Primeiros instantes:", trials[:10])
print(events[:5], len(events))
print(y[:10], len(y))

Primeiros instantes: [6000 6026 6472 6914 7354 7799 8240 8683 9133 9570]
[[6000    7]
 [6026    4]
 [6472    4]
 [6914    4]
 [7354    3]] 567
[7 4 4 4 3 4 4 3 4 4] 567


In [15]:
# separando os dados em blocos de eventos/estímulos

data_f = list()

for nf in range(len(trials)):
    if nf < len(trials) - 1:
        data_f.append(X[trials[nf]:trials[nf + 1], :])
    else:
        # Último bloco até o fim de uma janela padrão de 2 segundos ou limite do sinal
        fim = min(trials[nf] + int(2 * fs), X.shape[0])
        data_f.append(X[trials[nf]:fim, :])

In [16]:
chs = dict()

for s in set([len(f) for f in data_f]):
    chs[s] = 0

for s in [len(f) for f in data_f]:
    chs[s] += 1

for k, v in chs.items():
    print(f'{k}: {v}', end=' | ')

400: 1 | 22: 3 | 23: 3 | 24: 7 | 25: 5 | 26: 5 | 27: 3 | 28: 1 | 433: 4 | 434: 5 | 435: 4 | 436: 21 | 437: 29 | 438: 20 | 439: 24 | 440: 47 | 441: 34 | 442: 29 | 443: 47 | 444: 32 | 445: 36 | 446: 61 | 447: 28 | 448: 19 | 449: 28 | 450: 32 | 5438: 1 | 451: 5 | 5445: 1 | 454: 2 | 453: 3 | 5446: 2 | 5449: 3 | 455: 1 | 452: 2 | 5448: 2 | 5451: 1 | 5455: 1 | 5456: 2 | 5457: 2 | 5458: 4 | 5459: 2 | 5461: 1 | 5462: 1 | 12001: 2 | 5442: 1 | 

In [17]:
# Padronizando a quantidade de amostras
data_f_padronizado = []

# Pega o tamanho mais frequente encontrado no dicionário chs
tamanhos_ordenados = sorted(chs.items(), key=lambda item: item[1], reverse=True)
n_amostras = tamanhos_ordenados[0][0] if len(tamanhos_ordenados) > 0 else int(2 * fs)
n_canais = X.shape[1]

for segmento in data_f:
    segmento = np.asarray(segmento)

    if segmento.size > 0 and segmento.ndim == 1:
        segmento = segmento.reshape(1, -1)

    if segmento.size == 0:
        segmento = np.zeros((n_amostras, n_canais), dtype=X.dtype)
    elif len(segmento) < n_amostras:
        repeticoes = int(np.ceil(n_amostras / len(segmento)))
        segmento = np.tile(segmento, (repeticoes, 1))[:n_amostras]
    else:
        segmento = segmento[:n_amostras]

    data_f_padronizado.append(segmento)

data_f = np.asarray(data_f_padronizado)
print("Formato de data_f:", data_f.shape)

Formato de data_f: (567, 446, 30)


In [18]:
# Repetindo o formato para o conjunto de labels

data_y = np.array(y)
print("Formato de data_y:", data_y.shape)

# salvamento dos dados em numpy arrays
np.save('data_f.npy', data_f)
np.save('data_y.npy', data_y)

Formato de data_y: (567,)
